#Introdution

** Description: **Lab tests are not a procedure, if something is observed with an expected resulting amount and unit then it should be a measurement. Phlebotomy is a procedure but so trivial that it tends to be rarely captured. It can be assumed that there is a phlebotomy procedure associated with many lab tests, therefore it is unnecessary to add them as separate procedures. If the user finds the same procedure over concurrent days, it is assumed those records are part of a procedure lasting more than a day. This logic is in lieu of the procedure_end_date, which will be added in a future version of the CDM.

In [ ]:
#Data analysis

In [4]:
import pandas as pd

In [5]:
procedure_occurrence = pd.read_csv("../data/procedure_occurrence_202611080931.csv")

In [11]:
visit_occurrence = pd.read_csv("../data/visit_occurrence_202611061448.csv")
person = pd.read_csv("../data/person_202611061444.csv")

In [4]:
#Number of columns and rows
rows, column = (procedure_occurrence.shape)
print(f"rows: {rows} \ncolumn {column}")

total_visit = len(procedure_occurrence)
print(f"total_visit: {total_visit}")

rows: 47042032 
column 5
total_visit: 47042032


In [6]:
#Null Values
null_amount = procedure_occurrence.isnull().sum()
null_porcent = (null_amount/total_visit)*100

summary_null = pd.DataFrame({
    'Missing values': null_amount,
    'Porcent (%)': null_porcent
})

print(summary_null.sort_values(by='Missing values', ascending=False))

                         Missing values  Porcent (%)
procedure_date                 47042032   100.000000
visit_occurrence_id              370712     0.788044
procedure_occurrence_id               0     0.000000
person_id                             0     0.000000
procedure_concept_id                  0     0.000000


In [ ]:
#Duplicate values
list_columns = procedure_occurrence.columns.to_list()

for i in list_columns:
    num_duplicate = procedure_occurrence[i].duplicated().sum()

    if num_duplicate > 0:
        print(f"The column {i} has duplicate values ({num_duplicate} in total)")
    else:
        print(f"The column {i} has no duplicate")

#Duplicate total per rows
duplicate_records = procedure_occurrence.duplicated().sum()
print("Total of total duplicate per rows: ", duplicate_records)

The column procedure_occurrence_id has no duplicate
The column person_id has duplicate values (46888604 in total)
The column visit_occurrence_id has duplicate values (46379249 in total)
The column procedure_concept_id has duplicate values (47041874 in total)
The column procedure_date has duplicate values (47042031 in total)
Total of total duplicate per rows:  0


In [10]:
#Check if the visit_occurrence_id exist in visit_occurrence
values_exist = (visit_occurrence["visit_occurrence_id"].isin(procedure_occurrence["visit_occurrence_id"])).sum()
values_not_exist = (~visit_occurrence["visit_occurrence_id"].isin(procedure_occurrence["visit_occurrence_id"])).sum()
print(f"Numbers of IDs that don't exist in visit_occurrence: {values_exist}")
print(f"Numbers of IDs that exist in visit_occurrence: {values_not_exist}")


Numbers of IDs that don't exist in visit_occurrence: 662629
Numbers of IDs that exist in visit_occurrence: 1043372


In [12]:
#Check if the person_id exist in person
values_exist = (person["person_id"].isin(procedure_occurrence["visit_occurrence_id"])).sum()
values_not_exist = (~person["person_id"].isin(procedure_occurrence["visit_occurrence_id"])).sum()
print(f"Numbers of IDs that don't exist in visit_occurrence: {values_exist}")
print(f"Numbers of IDs that exist in visit_occurrence: {values_not_exist}")


Numbers of IDs that don't exist in visit_occurrence: 112402
Numbers of IDs that exist in visit_occurrence: 799803


In [14]:
#Conditions regarding visits
procedure_per_visit = procedure_occurrence["visit_occurrence_id"].value_counts()
procedure_distribution = procedure_per_visit.value_counts().sort_index()
print(procedure_distribution)

#Maximun condition
print(f"The higher maximun condition is: {procedure_per_visit.max()}")
print(f"The value most apper is: {procedure_per_visit.idxmax()}")


count
1           620
2        123393
3        119259
4         53493
5         44714
          ...  
39508         1
39780         1
41769         1
51304         1
53636         1
Name: count, Length: 2893, dtype: int64
The higher maximun condition is: 53636
The value most apper is: 3093796.0


In [ ]:
#Ten most common values in procedure_concept_id
value_counts_result = procedure_occurrence['procedure_concept_id'].value_counts().head(10)

top_10_df = procedure_occurrence['procedure_concept_id'].value_counts().head(10).reset_index()
top_10_df.columns = ['procedure_concept_id', 'frequency']
total_procedure_occurence = len(procedure_occurrence)
top_10_df['percentage'] = (top_10_df['frequency'] / total_procedure_occurence) * 100
print(top_10_df)
top_10_df["concept_name"]= ("Glucose measurement, blood",
                            "Hemogram, automated, with RBC, WBC, Hgb, Hct, Indices, Platelet count, and automated complete WBC differential",
                            "Emergency department Medical records",
                            "Specialized medical examination",
                            "Emergency department Consult note",
                            "Creatinine measurement",
                            "Urea nitrogen [Moles/volume] in Serum or Plasma",
                            "Serum potassium measurement",
                            "Sodium [Moles/volume] in Serum or Plasma",
                            "Chloride [Moles/volume] in Serum or Plasma",  
)

   procedure_concept_id  frequency  percentage
0               4144235    4812788   10.230825
1               4016928    2953663    6.278774
2               3003184    2951054    6.273228
3               4194672    2827605    6.010805
4               3040972    2500072    5.314549
5               4324383    2368474    5.034804
6               3024641    2342467    4.979519
7               4154489    1982374    4.214048
8               3019550    1898733    4.036248
9               3014576    1387334    2.949137


In [ ]:
### PROCEDURE_OCCURRENCE – Key Findings

1. All procedure date fields contain null values, making temporal analysis of procedures impossible.

2. Among the ten most frequent `procedure_concept_id` values, eight belong to the `Measurement` domain, one belongs to the `Note` domain, and only one belongs to the `Procedure` domain, according to OHDSI Athena.

3. All ten concepts are classified as standard concepts (`Standard Concept = S`). However, nine of them belong to domains other than `Procedure`, suggesting potential inconsistencies in concept mapping or domain assignment.

4. These inconsistencies limit the reliability of clinical interpretations based on `procedure_concept_id`.

5. Although procedure records can be linked to patients and visits, the absence of dates and the identified domain inconsistencies limit their usefulness for temporal analysis and demand forecasting.